Ad Hoc Ray Cluster - Not Managed 
- manually launch a Ray head node on the Databricks driver.
- manually launch 4 Ray workers, each in its own Spark executor.
- Each Spark executor was started with num_gpus_per_worker=4.
- Therefore, the total cluster resources = 16 GPUs (4 × 4).

Ray directly used those resources, and  explicitly scheduled 4 remote GPU tasks.
Everything stayed inside that one cluster  spawned — no scheduler above Ray

In [0]:
# --- Imports & setup ---
import ray
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

# Clean up any existing Ray cluster
try:
    shutdown_ray_cluster()
except Exception:
    pass

# --- Start a new Ray-on-Spark cluster ---
cluster_handle, ray_address = setup_ray_cluster(
    min_worker_nodes=4,
    max_worker_nodes=4,
    num_cpus_per_worker=4,   # adjust to match your Spark worker config
    num_gpus_per_worker=4,
    num_cpus_head_node=2,
    num_gpus_head_node=0,
    collect_log_to_path="/dbfs/ray_logs"  # optional
)

# Connect Ray runtime
ray.init(address=ray_address, ignore_reinit_error=True)

# --- Show overall cluster resources ---
print("=== Cluster Resources (Total) ===")
for k, v in ray.cluster_resources().items():
    print(f"{k:<15}: {v}")

# --- Show per-node resource summary ---
print("\n=== Per-Node Resources ===")
for node in ray.nodes():
    addr = node["NodeManagerAddress"]
    res = node["Resources"]
    print(f"Node {addr:<15} | CPUs: {res.get('CPU', 0):<3} | GPUs: {res.get('GPU', 0):<3}")

# --- Test GPU usage with a remote task ---
@ray.remote(num_gpus=1)
def check_gpu():
    import torch
    return {
        "host": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA",
        "count": torch.cuda.device_count(),
    }

results = ray.get([check_gpu.remote() for _ in range(4)])
print("\n=== Remote GPU Visibility Check ===")
for i, r in enumerate(results):
    print(f"Task {i}: {r}")

# --- Summary check ---
total_gpus = ray.cluster_resources().get("GPU", 0)
print(f"\nTotal GPUs detected by Ray: {int(total_gpus)}")

# Optional: automatically assert expected GPU count
expected = 16
if int(total_gpus) != expected:
    raise RuntimeError(f"Expected {expected} GPUs but Ray reports {total_gpus}")

# Uncomment when you're done
shutdown_ray_cluster()

Result

- Ray saw all 16 GPUs
- Each ray.remote() call mapped cleanly to a worker
- No higher-level resource manager intervened


###  **Hybrid Ray-on-Spark + Ray Train Setup**

This notebook uses **Ray-on-Spark** to launch a full Ray cluster across all GPU nodes, and then attaches a **Ray Train TorchTrainer** job to that same cluster.  

Normally, when you call `TorchTrainer` directly on Databricks, Ray Train creates a new internal Ray job that only “sees” the driver’s local GPUs.  
That’s why earlier TorchTrainer runs failed to detect all 16 GPUs — it wasn’t aware of the multi-node Ray cluster Spark had provisioned.

By first calling:

    cluster_handle, ray_address = setup_ray_cluster(...)
    ray.init(address=ray_address)

we create a **true multi-node Ray cluster** (4 nodes × 4 GPUs = 16 GPUs total).  
Then, when we launch `TorchTrainer`, it connects to that existing Ray cluster rather than spinning up its own mini-cluster.  

This hybrid pattern gives us:
- Full access to all GPUs across all nodes  
- Correct process-group initialization for distributed data-parallel (DDP) training  
- Accurate visibility into node topology and per-worker GPU assignments  
- Seamless use of Databricks Spark scheduling for Ray worker placement  

The **Ray-on-Spark layer provides the distributed cluster**, and **Ray Train runs the coordinated training job inside it**, ensuring consistent multi-GPU, multi-node scaling in Databricks.

Start the full Ray cluster

You *have to run this* in order to get the cluster to work across all nodes


In [0]:
%pip install ray[default]==2.33.0  
%pip install pytorch-lightning
dbutils.library.restartPython()

In [0]:

from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster
import ray, time
ray.shutdown()
# Clean up any previous cluster
try:
    shutdown_ray_cluster()
except Exception:
    pass

# Spin up 4 Spark executors, each exposing 4 GPUs  →  16 GPUs total
cluster_handle, ray_address = setup_ray_cluster(
    min_worker_nodes=4,
    max_worker_nodes=4,
    num_cpus_per_worker=4,
    num_gpus_per_worker=4,
    num_cpus_head_node=2,
    num_gpus_head_node=0,
    collect_log_to_path="/dbfs/ray_logs"
)

# Attach to that same cluster
ray.init(address=ray_address, ignore_reinit_error=True)
time.sleep(5)

print("=== Ray cluster resources ===")
print(ray.cluster_resources())

Run TorchTrainer inside that cluster

In [0]:
import os
import socket
import torch
import torch.nn as nn
import torch.optim as optim
from ray.train.torch import TorchTrainer, prepare_model
from ray.train import ScalingConfig
from ray import train

def train_func(config):
    import ray

    ctx = train.get_context()
    world_rank = ctx.get_world_rank()
    node_rank = ctx.get_node_rank()
    local_rank = int(os.environ.get("LOCAL_RANK", -1))
    node_ip = ray._private.services.get_node_ip_address()

    # Actual Ray-assigned GPUs (via CUDA_VISIBLE_DEVICES)
    gpu_ids = os.environ.get("CUDA_VISIBLE_DEVICES", "")
    gpu_count = torch.cuda.device_count()
    gpu_name = torch.cuda.get_device_name(0) if gpu_count > 0 else "CPU only"

    print(
        f"[Worker {world_rank:02d}] Node={node_ip} "
        f"NodeRank={node_rank} LocalRank={local_rank} "
        f"CUDA_VISIBLE_DEVICES={gpu_ids} ({gpu_name})"
    )

    # Model setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 10))
    model = prepare_model(model)  # handles DDP + device placement
    opt = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    # Tiny dummy loop
    for step in range(3):
        x = torch.randn(128, 32, device=device)
        y = torch.randint(0, 10, (128,), device=device)
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
        if world_rank == 0:
            print(f"Step {step} | Loss={loss.item():.4f}")

    # Report metrics to Ray dashboard
    train.report({
        "final_loss": loss.item(),
        "node_ip": node_ip,
        "gpu_ids": gpu_ids,
        "gpu_name": gpu_name,
        "world_rank": world_rank
    })


# --- Scaling config: one worker per GPU ---
scaling = ScalingConfig(
    num_workers=16,  # 4 nodes × 4 GPUs
    use_gpu=True,
)

# --- Trainer setup ---
trainer = TorchTrainer(
    train_loop_per_worker=train_func,
    scaling_config=scaling,
)

result = trainer.fit()

print("\n=== Training Complete ===")
print(f"Loss: {result.metrics['final_loss']:.4f}")
print("Example worker metrics:")
for k, v in list(result.metrics.items())[:5]:
    print(f"  {k}: {v}")

This is pytorch lightning code

In [0]:
# --- PyTorch Lightning on Ray using Ray's native PL integration (no index errors) ---
import os, torch, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from pytorch_lightning import LightningModule, Trainer
from ray.train import ScalingConfig, RunConfig, FailureConfig
from ray.train.torch import TorchTrainer
from ray.train.lightning import RayDDPStrategy, RayLightningEnvironment
from ray import train
import ray

# Quiet noisy libs
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("FLASH_ATTENTION_DISABLE", "1")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "true")
os.environ.setdefault("DISABLE_TRANSFORMERS_AV", "1")
# Conservative NCCL defaults for multi-node envs
os.environ.setdefault("TORCH_NCCL_ASYNC_ERROR_HANDLING", "1")
os.environ.setdefault("NCCL_BLOCKING_WAIT", "1")
os.environ.setdefault("NCCL_P2P_DISABLE", "1")
os.environ.setdefault("NCCL_IB_DISABLE", "1")

class TinyLightning(LightningModule):
    def __init__(self):
        super().__init__()
        self.layer1 = torch.nn.Linear(32, 64)
        self.layer2 = torch.nn.Linear(64, 10)
    def forward(self, x):
        return self.layer2(torch.relu(self.layer1(x)))
    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = F.cross_entropy(self(x), y)
        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

def train_func(config):
    import pytorch_lightning as pl
    import ray as _ray

    # Nice perf hint for A10G tensor cores
    try: torch.set_float32_matmul_precision("high")
    except Exception: pass

    rank = train.get_context().get_world_rank()
    node_ip = _ray._private.services.get_node_ip_address()
    gpu_env = os.environ.get("CUDA_VISIBLE_DEVICES", "")
    print(f"[Worker {rank:02d}] Node={node_ip} CVD={gpu_env} cuda_count={torch.cuda.device_count()}")

    # Tiny synthetic dataset
    x = torch.randn(4096, 32)
    y = torch.randint(0, 10, (4096,))
    loader = DataLoader(
        TensorDataset(x, y),
        batch_size=128,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=False,
    )

    model = TinyLightning()

    # Use Ray's PL integration so ranks/devices map correctly
    strategy = RayDDPStrategy()                     # <-- key change
    env = RayLightningEnvironment()                 # <-- key change

    trainer = pl.Trainer(
        accelerator="gpu",
        devices=1,           # one GPU per Ray worker
        num_nodes=1,         # PL doesn't re-interpret nodes; Ray handles world size
        strategy=strategy,
        plugins=[env],
        max_epochs=2,
        logger=False,
        enable_progress_bar=(rank == 0),
        inference_mode=False,
    )

    trainer.fit(model, loader)
    loss = float(trainer.callback_metrics.get("train_loss", 0.0))
    train.report({"final_loss": loss, "node_ip": node_ip, "gpu_env": gpu_env})

# Scale to all GPUs Ray sees (12 or 16)
total_gpus = int(ray.cluster_resources().get("GPU", 0))
print(f"=== Ray reports {total_gpus} GPUs ===")
assert total_gpus > 0, "Ray sees 0 GPUs — recheck cluster setup."

scaling = ScalingConfig(num_workers=total_gpus, use_gpu=True)
run_cfg = RunConfig(failure_config=FailureConfig(max_failures=3))  # auto-retry transient hiccups

trainer = TorchTrainer(
    train_loop_per_worker=train_func,
    scaling_config=scaling,
    run_config=run_cfg,
)

result = trainer.fit()
print("\n=== ✅ Lightning on Ray: training complete ===")
print(f"Loss: {result.metrics['final_loss']:.4f}")
print(f"Node: {result.metrics.get('node_ip')}")
print(f"CVD:  {result.metrics.get('gpu_env')}")